## Graph maker

### 1 - Import Libraries

In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import webbrowser
import import_ipynb

In [7]:
# indicator_func(df, to_calculate, timeframe, weighter=False, correlator=False)
import Indicator_Calculator

### 2 - Import Data from Bitcoin Futures csv file and check the dataframe

In [8]:
df = pd.read_csv('bitcoin_futures_raw_data/futures_raw_data.csv')

In [9]:
df.tail()

,datetime,open_price,high_price,low_price,close_price,volume,buy_volume,transactions,buy_transactions,long_short_ratio,...,low_predicted_funding_rate,close_predicted_funding_rate,open_funding_rate,high_funding_rate,low_funding_rate,close_funding_rate,open_open_interest,high_open_interest,low_open_interest,close_open_interest
2206,1758931200,109588.5,109700.0,109021.9,109577.3,35808.071,17875.676,342032,168645.0,1.7917,...,-0.000577,0.003332,-0.000010,0.007821,-0.000010,0.007821,84919.760,85127.114,84517.446,84550.888
2207,1759017600,109577.2,112300.0,109136.5,112119.6,77220.071,39810.148,614121,314616.0,1.7824,...,0.001616,0.002264,0.003314,0.005261,0.003314,0.005261,84550.926,86224.886,84418.636,86090.117
2208,1759104000,112119.7,114377.2,111501.0,114257.1,125365.977,63032.434,983709,487959.0,1.4900,...,-0.001262,0.002353,0.002312,0.002312,0.000032,0.000032,86089.696,88987.440,85641.969,88865.851
2209,1759190400,114257.1,114800.0,112615.3,113988.8,119261.073,60303.437,1063050,529369.0,1.0227,...,0.001904,0.003410,0.002353,0.005893,0.002353,0.002584,88864.528,90374.025,87585.658,88524.384
2210,1759276800,113988.7,114669.0,113899.4,114450.1,19162.092,9376.931,154609,74069.0,1.0683,...,0.001193,0.003825,0.003355,0.003355,0.003355,0.003355,88524.384,89769.610,88326.802,89763.968


In [10]:
df.columns

Index(['datetime', 'open_price', 'high_price', 'low_price', 'close_price',
       'volume', 'buy_volume', 'transactions', 'buy_transactions',
       'long_short_ratio', 'long_long_short_ratio', 's_long_short_ratio',
       'long_liquidation', 'short_liquidation', 'open_predicted_funding_rate',
       'high_predicted_funding_rate', 'low_predicted_funding_rate',
       'close_predicted_funding_rate', 'open_funding_rate',
       'high_funding_rate', 'low_funding_rate', 'close_funding_rate',
       'open_open_interest', 'high_open_interest', 'low_open_interest',
       'close_open_interest'],
      dtype='object')

### 3 - Calculate some indicator to test

### 4 - Create the Graph Function

In [ ]:
def nice_graph_maker(df, start, end, indicator_top, indicator_bot, timeframe_short, timeframe_long, colorby= 'close_price'):

    # gives the begining and end of the dataframe
    df = df[(df["datetime"] > start) & (df["datetime"] < end)]
    
    # list of thing to calculate the indicators
    to_calculate_list = ['close_price', indicator_top, indicator_bot]

    df_indicators_short = df[['datetime']]
    df_indicators_long = df[['datetime']]

    for t_c in to_calculate_list:
        if 'price' in t_c or 'interest' in t_c:
            df_indicators_short =  df_indicators_short.merge(Indicator_Calculator.indicator_func(df, t_c, timeframe_short, weighter='volume'), how='left', on='datetime')
            df_indicators_long =  df_indicators_long.merge(Indicator_Calculator.indicator_func(df, t_c, timeframe_long, weighter='volume'), how='left', on='datetime')
        else:
            df_indicators_short =  df_indicators_short.merge(Indicator_Calculator.indicator_func(df, t_c, timeframe_short), how='left', on='datetime')
            df_indicators_long =  df_indicators_long.merge(Indicator_Calculator.indicator_func(df, t_c, timeframe_long), how='left', on='datetime')

    # Constroi a base dos gráficos
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        row_heights=[0.25, 0.5, 0.25], 
        vertical_spacing=0.02
    )

    ####################################################################################################################################################################
    # (TOP indicator)
    # (Função)

    ####################################################################################################################################################################
    # (PREÇO)
    # (Função)
    def nice_candle_maker(name, condition, color, row):
        
        condition = condition[(condition["datetime"] > start) & (condition["datetime"] < end)]
                
        fig.add_trace(
            go.Candlestick(
                x=condition["datetime"],
                open=condition["open_price"],
                close=condition["close_price"],
                high=condition["high_price"],
                low=condition["low_price"],
                increasing=dict(
                                fillcolor=color,
                                line=dict(color=color)
                                ),
                decreasing=dict(
                                fillcolor=color,
                                line=dict(color=color)
                                ),
                name=name
            ),
            row=row,
            col=1         
        )
    # create new column with the quartile of z_score    
    df['quartile'] = pd.qcut(df_indicators_short[f'{colorby}_{timeframe_short}_z_score'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

    price_condition_dict = {
        'Quartile (75-100)' : [df[df.quantile == 'Q4'], 'rgb(197, 17, 98)'],
        'Quartile (50-75)' : [df[df.quantile == 'Q4'], 'rgb(229, 57, 53)'],
        'Quartile (25-50)' : [df[df.quantile == 'Q4'], 'rgb(77, 182, 172)'],
        'Quartile (0-25)' : [df[df.quantile == 'Q4'], 'rgb(144, 164, 174)'],
    }

    # (Call)    
    for name, (condition, color) in price_condition_dict.items():
        nice_candle_maker(name, condition, color, 2)

    ####################################################################################################################################################################
    # (Médias Móveis e Bandas de Bollinger)
    # (Função)
    def draw_ma(name, ma, color, width, row):
    
        fig.add_trace(
            go.Scatter(
                x=df["t"],
                y=df[ma],
                line=dict(color=color, width=width),
                name=name,
                showlegend=False
                ),
            row=row,
            col=1   
        )
    
    line_dict = {
        f'Price {timeframe_long} MA': [f'close_price_{timeframe_long}_ma', 'rgb(255, 213, 79)', 1, 2],
        f'Price {timeframe_long} Upper Band': [(f'close_price_{timeframe_long}_ma' + 2 * f'close_price_{timeframe_long}_std'), 'rgb(48, 63, 159)', 1, 2],
        f'Price {timeframe_long} Lower Band': [(f'close_price_{timeframe_long}_ma' - 2 * f'close_price_{timeframe_long}_std'), 'rgb(48, 63, 159)', 1, 2],
        indicator_top.replace('_', '').upper() : [f'{indicator_top}_{timeframe_long}_ma', 'rgb(255, 213, 79)', 1, 1],
        indicator_bot.replace('_', '').upper() : [f'{indicator_bot}_{timeframe_long}_ma', 'rgb(255, 213, 79)', 1, 2]
    }

    # (Call)    
    for name, (ma, color, width, row) in line_dict.items():
        draw_ma(name, ma, color, width, row)


